# Read Records against Their Sentences

| Item | Description |
|---|---|
| Goal | Read a sample of the assembled records next to the sentences they cite. Because the model cites sentence numbers, what is printed is exactly what it saw and cited; the check is about reading quality, not existence. Three samples: twenty records drawn at random, ten per stance, and five flagged by the lint. |
| Input | `stance_records.parquet` and `member_turns.parquet` from the T2.2 Processed folder; hearing titles from `hearing_summaries.parquet` of T1.2. |
| Output | None. This notebook writes no file. The printed samples are the record of the check. |
| Reading | In the August reading, roughly 85 to 90 percent of sampled records were clean on all three fields. Three rare defects were noted: a stance aimed at the harm rather than the reform, vague topics such as "agencies", and a record whose cited sentence alone does not carry the topic. |
| Run | First run 2026-08-22. Run again on 2026-10-31 on the stored tables; the seeds are fixed, so the same records are printed. |


## Step 1. Import packages


In [1]:
from pathlib import Path

import pandas as pd


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.2 - Track 2 - Legislator Stance Extraction")
NOTEBOOK_NAME = "T2.2.5 - Check - Read Records against Their Sentences"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
INPUT_FOLDER = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T1 - Text Summarization/T1.2 - Process Entire Sample Congressional Hearings" / "Processed"
SEED = 7
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Load the records, the sentences, and the hearing titles


In [4]:
records = pd.read_parquet(PROCESSED_FOLDER / "stance_records.parquet")
turns = pd.read_parquet(PROCESSED_FOLDER / "member_turns.parquet", columns=["packageId", "turn_idx", "sentences"])
titles = pd.read_parquet(INPUT_FOLDER / "hearing_summaries.parquet", columns=["packageId", "title"]).set_index("packageId")["title"]
sent = {(p, t): s for p, t, s in zip(turns["packageId"], turns["turn_idx"], turns["sentences"])}
print(f"{len(records):,} records, {len(sent):,} turns loaded")


def show(df, header):
    print("=" * 78)
    print(header)
    print("=" * 78)
    for r in df.itertuples():
        who = f"{r.honorific or ''} {r.surname or ''}".strip()
        evidence = " ".join(sent[(r.packageId, r.turn_idx)][i - 1] for i in r.sentences)
        print(f"\n--- {r.packageId} ({r.year} {r.chamber}), {who}{' [chair]' if r.is_chair else ''}")
        print(f"    hearing : {str(titles.get(r.packageId, ''))[:90]}")
        print(f"    evidence: {evidence[:400]}")
        print(f"    topic   : {r.topic}")
        print(f"    stance  : {r.stance.upper()}" + (f"   [lint: {r.topic_lint}]" if r.topic_lint else ""))


32,385 records, 221,311 turns loaded


## Step 4. Read twenty records drawn at random


In [5]:
show(records.sample(20, random_state=SEED), "RANDOM SAMPLE (n=20)")


RANDOM SAMPLE (n=20)

--- CHRG-107shrg88291 (2001 SENATE), Senator Cleland
    hearing : ELECTION REFORM
    evidence: I am an author of legislation to provide a knock-out punch to the punch card system, not by putting a burden on counties, but by putting some federal funds to a third of the precincts in America that have that system, or are now afflicted with that system.
    topic   : federal funds for punch card system reform
    stance  : SUPPORT

--- CHRG-117shrg55821 (2022 SENATE), Senator Moran
    hearing : NOMINATION TO THE AMTRAK BOARD OF DIRECTORS AND THE SURFACE TRANSPORTATION BOARD
    evidence: And I am disappointed a bit in the response that I received or that several of us received from you in regard to your commitment to long distance passenger service.
    topic   : long distance passenger service
    stance  : CONCERN

--- CHRG-106shrg79713 (2000 SENATE), Senator McCain
    hearing : S. 2340, AMATEUR SPORTS INTEGRITY ACT AND GAMBLING IN AMATEUR SPORTS
    evidence: I

## Step 5. Read ten records of each stance

Each should read as "the member supports [topic]" or "the member is concerned about [topic]".


In [6]:
show(records[records["stance"] == "support"].sample(10, random_state=SEED), "SUPPORT (n=10)")
show(records[records["stance"] == "concern"].sample(10, random_state=SEED), "CONCERN (n=10)")


SUPPORT (n=10)

--- CHRG-116shrg54131 (2020 SENATE), The Chairman [chair]
    hearing : DOES SECTION 230'S SWEEPING IMMUNITY ENABLE BIG TECH BAD BEHAVIOR?
    evidence: This legislation would address the challenges we have discussed while still leaving fundamentals of Section 230 in place.
    topic   : Section 230
    stance  : SUPPORT

--- CHRG-113shrg88868 (2013 SENATE), Senator Blumenthal
    hearing : CRAFTING A SUCCESSFUL INCENTIVE AUCTION: STAKEHOLDERS' PERSPECTIVES
    evidence: And so I am hoping, and I am going to ask this question of Mr. Epstein, that the FCC will pursue a spectrum policy that best enables competitive forces to benefit consumers and will adopt the Department of Justice recommendations or the kinds of screens or caps that limit the amount of spectrum that any one company can have, simply so that there can be more competition.
    topic   : spectrum policy
    stance  : SUPPORT

--- CHRG-115shrg28382 (2017 SENATE), Senator Hassan
    hearing : THE PROMISES AND

## Step 6. Read five records the lint flagged

Most flags are proper names that contain a directional word, such as the "Stop Business with Terrorists Act", not rule violations.


In [7]:
show(records[records["topic_lint"].notna()].sample(5, random_state=SEED), "LINT-FLAGGED (n=5)")


LINT-FLAGGED (n=5)

--- CHRG-110shrg35959 (2007 SENATE), Senator Lautenberg
    hearing : HALLIBURTON AND U.S. BUSINESS TIES TO IRAN
    evidence: Mr. Chairman, we've got to stop this behavior. That's why I reintroduced my bill, Stop Business with Terrorists Act last week. The bill will close the loophole that Halliburton has exploited in order to do business with Iran.
    topic   : Stop Business with Terrorists Act
    stance  : SUPPORT   [lint: Stop]

--- CHRG-110shrg77867 (2007 SENATE), Senator Vitter
    hearing : POLICY IMPLICATIONS OF PHARMACEUTICAL IMPORTATION FOR U.S. CONSUMERS
    evidence: I'm happy to say we put an end to that practice--first, with this amendment on an appropriations bill, and that led to the current USTR abandoning, and stating very clearly to me and others that she would abandon this practice of trying to negotiate those provisions in trade deals.
    topic   : practice of inserting anti-reimportation language through bilateral trade agreements
    stance